## Classify emergent detections from STA/LTA on the basis of waveform characteristics
- runs in parallel over the list of detections

In [1]:
import os
import pickle
import glob
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import datetime
import numpy as np
import obspy
from obspy.clients.fdsn.client import Client 
import obspy
import pandas as pd
import scipy.ndimage
import geopy.distance
import random
client = Client('IRIS')
import scipy
import seaborn as sn
import dask
from dask.diagnostics import ProgressBar
import time

## Pull in emergent detections from STA/LTA triggering

In [2]:
# For all:
file_name='mt_classifications/OBS141.pickle'
with open(file_name,'rb') as handle:
    detections = pickle.load(handle)

In [3]:
t1 = obspy.UTCDateTime("2003-06-23T00:00:00.000")
t2 = obspy.UTCDateTime("2004-05-03T00:00:00.000")

network = 'YY'
station = 'OBS14'
channel = 'HH2'

samp_rate = 200

In [4]:
# Filter the detections based on time, if desired!
diff = t2 - t1
days = diff / (60 * 60 * 24)
print(days)

t_keep = [i for i,e in enumerate(detections) if (e[0]>t1) & (e[0]<t2)]
detections = [detections[i] for i in t_keep]

315.0


## Define classification functions

In [5]:
def pick_peaks_welch(trace,sampling_rate,nperseg_multiple,microseism_cutoff=True):
    """
    Estimate power spectra of a trace using Welch's method
    Pick peaks within the spectra!
    
    INPUTS:
    trace = obspy object, waveform
    sampling_rate = sampling rate of trace
    nperseg_multiple = length of each segment used to construct the Welch spectrum
    microseism_cutoff = Bool, whether or not to cut off the lower end of the spectrum to avoid the microseism
    
    OUTPUTS:
    f = frequencies of the spectra
    Pxx_den = associated power at each frequency, in decibels
    peak_ind = index of peaks within the spectra (f and Pxx_den), if found
    peaks = picked peak object from scipy
    median_power = median power of spectra from 20-80 Hz in decibels
    """
    
    
    fs = sampling_rate
    x = trace.data
    nperseg = fs * nperseg_multiple
    
    f,Pxx_den = scipy.signal.welch(x,fs,nperseg=nperseg)
    if microseism_cutoff is True:
        f = f[4:]
        Pxx_den = Pxx_den[4:]
        
    Pxx_den = [10*np.log10(d) for d in Pxx_den]
    median_power = np.median(Pxx_den[20:80])
    
    peaks = scipy.signal.find_peaks(Pxx_den,threshold =median_power*5,prominence=10) 
    peak_ind = peaks[0]
    
    return(f,Pxx_den,peak_ind,peaks,median_power)

In [6]:
def apply_gaussian(filtered_data,samp_rate,gaussian_width=5):
    """
    Smooth waveform using a gaussian window
    
    INPUTS
    filtered_data = filtered numpy array of seismic data (from an obspy trace)
    samp_rate = sampling rate of data
    gaussian width = width of Gaussian window in seconds
    
    OUTPUTS
    smoothed_window = smoothed numpy array of seismic data
    """
    
    # Square data
    data = filtered_data**2
    
    gaussian_radius = int((gaussian_width * samp_rate)/2)
    smoothed_window=scipy.ndimage.gaussian_filter1d(data,sigma=gaussian_radius/4,radius=gaussian_radius)
    
    return smoothed_window

In [7]:
def ship_noise_classifier(trace,sampling_rate):
    """
    Check whether detection likely includes ship noise in the form of a spectral peak
    
    INPUTS
    trace = obspy trace object
    sampling_rate = sample rate of trace
    
    OUTPUTS
    ship_classifier = number of peaks in the spectra. If any exist, ship noise is likely!
    """
    
    
    # Pick peaks on the smoothed spectrum of the trace (nperseg multiple = 1)
    f,Pxx_den,peak_ind,peak_details,median_power = pick_peaks_welch(trace,sampling_rate,1,microseism_cutoff=True)
    
    if len(peak_ind)==0:
        ship_classifier = 0
    else:
        ship_classifier = len(peak_ind)

    
    return ship_classifier

1. Calculate whether the detection likely includes ship noise (whether there are peaks in the Welch spectra)
2. Perform 15 s Gaussian smoothing on filtered (4-15 Hz) waveform, calculate number of peaks with prominence > 0.1
- 1 peak indicates T-phase, > 1 peak is consistent with tremor
3. Calculate frequency ratio (ratio between power in 5-10 and 10-15 Hz) with 30 s padding on either side
- Frequency ratio > 100 is consistent with tectonic tremor

In [8]:
def classify_detection(t, network, station, channel, samp_rate, filepath):
    """
    INPUTS 
    t = tuple of UTCDatetime, with on and off trigger time of detection
    network = string
    station = string
    channel = string
    samp_rate = sampling rate of channel
    filepath = directory to save classification information to
    
    
    OUTPUTS
    Writes to file!
    t = on and off times of detection
    num_waveform_peaks = number of peaks with prominence > 0 in gaussian smoothed 4-15 Hz filtered waveform; more than 1 indicates a T-phase
    ship_classifier = number of peaks in the Welch spectra, existence of peaks indicates ship noise
    freq_ratio_welch = ratio between normalized decibels of Welch spectrum for 5-10 and 10-15 Hz. Values > 100 indicate tectonic tremor
    max_amplitude = maximum amplitude of filtered waveform
    """
    '''
    file_name = filepath +  station + '_' + str(t[0]).split('.')[0] + '.pickle'
    if os.path.isfile(file_name)==True:
        return
    '''

    day_str = t[0].strftime("%Y-%m-%d")
    file_name = os.path.join(filepath, f"{station}_{day_str}.pickle")
        
# get waveform
    def get_trace(t1, t2, attach_response=True):
        time.sleep(3)
        st = client.get_waveforms(network, station, "*", channel, t1 - 5, t2 + 5,attach_response=attach_response)
        st.resample(samp_rate).merge(fill_value="interpolate")
        return st[0]
    
# ship noise
    pad = 0
    t1 = t[0] - pad
    t2 = t[1] + pad

    try:
        st1 = get_trace(t1, t2, attach_response=True)

        # check response and continue if none
        if not hasattr(st1.stats, "response"):
            return 2
        resp = st1.stats.response
        if resp is None or resp.instrument_sensitivity is None:
            return 3

        sens = resp.instrument_sensitivity.value
        st1.data = st1.data / sens  # m/s
        st1.trim(starttime=t1, endtime=t2)

        ship_classifier = ship_noise_classifier(st1, samp_rate)

    except Exception:
        return 4
    
# count waveform peaks
    try:
        st1 = get_trace(t1, t2, attach_response=True)
        st1.filter("bandpass", freqmin=3, freqmax=10)
        try:
            st1.remove_response()
        except Exception:
            return 5

        st1.trim(starttime=t1, endtime=t2)

        max_amplitude = np.max(np.abs(st1.data))

        smoothed_window = apply_gaussian(st1.data, samp_rate, gaussian_width=15)
        window_max = np.max(smoothed_window)
        if window_max == 0:
            return 6

        smoothed_window = smoothed_window / window_max
        peaks = scipy.signal.find_peaks(smoothed_window, prominence=0.1)
        num_waveform_peaks = len(peaks[0])

    except Exception:
        return 7

# calculate frequency ratio
    try:
        pad = 30
        t1 = t[0] - pad
        t2 = t[1] + pad

        st2 = get_trace(t1, t2, attach_response=True)

        if not hasattr(st2.stats, "response"):
            return 8
        resp = st2.stats.response
        if resp is None or resp.instrument_sensitivity is None:
            return 9

        sens = resp.instrument_sensitivity.value
        st2.data = st2.data / sens
        st2.trim(starttime=t1, endtime=t2)

        f, Pxx_den, peak_ind, peak_details, median_power = pick_peaks_welch(st2, samp_rate, 5, microseism_cutoff=False)

        normalized_power = Pxx_den
        freq_ratio_welch = (10 ** (np.median(normalized_power[25:50]) / 10) / 10 ** (np.median(normalized_power[50:75]) / 10))

    except Exception as e:
        return 10

    print(num_waveform_peaks)
# write results to file
    with open(file_name, "wb") as handle:
        pickle.dump([t, station, num_waveform_peaks,ship_classifier,freq_ratio_welch,max_amplitude],handle)

    return 0


## Loop in parallel
Note: simply overwrites files if they already exist

In [9]:
filepath = 'mt_classifications/'

In [10]:
st = client.get_waveforms(network, station, "*", channel, t1 - 5, t2 + 5,attach_response=True)
#st.resample(samp_rate).merge(fill_value="interpolate")
print(st[0])

YY.OBS14..HH2 | 2003-06-23T01:00:00.276000Z - 2003-06-23T02:00:00.268188Z | 128.0 Hz, 460800 samples


In [11]:
@dask.delayed(pure=False)
def loop_detections(t,network,station,channel,samp_rate,filepath):
    return classify_detection(t,network,station,channel,samp_rate,filepath)

In [12]:
lazy_results = [loop_detections(t,network,station,channel,samp_rate,filepath) for t in detections]

In [ ]:
with ProgressBar():
    results = dask.compute(lazy_results,num_workers=3)

[                                        ] | 0% Completed | 10.61 sms2
[                                        ] | 0% Completed | 10.81 s1
1
[                                        ] | 0% Completed | 20.98 s1
[                                        ] | 0% Completed | 21.19 s1
[                                        ] | 0% Completed | 21.30 s1
[                                        ] | 0% Completed | 31.23 s0
[                                        ] | 0% Completed | 31.66 s6
[                                        ] | 0% Completed | 31.76 s2
[                                        ] | 0% Completed | 41.54 s2
[                                        ] | 0% Completed | 41.75 s1
[                                        ] | 0% Completed | 42.07 s3
[                                        ] | 0% Completed | 51.75 s1
1
[                                        ] | 0% Completed | 52.19 s1
[                                        ] | 0% Completed | 62.06 s1
[                           

In [ ]:
results

## Pull in all saved files of classifications and save to one pickle file

In [ ]:
files = glob.glob(filepath+station+'_2'+'*')

classifications = []
for f in files:
    with open(f,'rb') as handle:
        classi = pickle.load(handle)
        classifications.append(classi)

pickle.dump(classifications,open('mt_classifications/'+station+'_classifications.pickle', 'wb'))

# Sort!
'''
times = [c[0][0] for c in classifications]
sort_ind = np.argsort(times)
classifications = [classifications[i] for i in sort_ind]
'''

In [ ]:
with open('mt_classifications/'+station+'_classifications.pickle', 'rb') as handle:
    data= pickle.load(handle)
    
#for i in range(len(data)):
#print(len(data[i]))
#print(data)